# 分析①


## データハンドリング入門：夏目漱石『夢十夜』の分析

+ データハンドリング
+ 単語の頻度の集計
+ TF-IDF


In [ ]:
# ライブラリの読み込み
library('dplyr')
library('stringr')
library('RMeCab')

In [ ]:
# 各自の環境による
getwd()

In [ ]:
yume = read.delim('data/textmining/yumejuya.tsv', header=T, sep='\t', stringsAsFactors=F, fileEncoding='utf8')

+ section: 第一夜から第十夜までが数字で
+ paragraph: 各話の段落番号
+ content: 本文


In [ ]:
yume %>% head()

In [ ]:
# 段落の長さの分布
# str_lengt: 文字列の長さ
yume[, 'content'] %>% str_length() %>% hist(breaks=25, xlab='Paragraph length', main='Histogram of paragraph length')

In [ ]:
# 各話ごとの段落の長さの分布を箱ひげ図で可視化する
# 段落の長さを格納する
yume['length'] = yume[, 'content'] %>% str_length()

In [ ]:
boxplot(length ~ section_id, data=yume, main='Paragraph length of each section')

In [ ]:
# 分析のため各話ごとに文章を結合する
# group_by: データを集約する
# summarise: データを特定の関数でまとめる
sections = yume %>% group_by(section_id) %>% summarise(text = paste0(content, collapse=''))
sections = as.data.frame(sections)


In [ ]:
# データの形状を確認する
dim(sections)

In [ ]:
colnames(sections)

In [ ]:
# 各夜の長さ
sections[, 'text'] %>% str_length() %>% barplot()

In [ ]:
# docMatrixDF: dataframeから単語を抽出

count_noun = docMatrixDF(sections[,'text'], pos=c('名詞'))

In [ ]:
count_noun %>% head()

In [ ]:
# 全体を集計する
freq_noun = count_noun %>% rowSums()

In [ ]:
# 全体を集計する
freq_noun %>% sort(decreasing=T) %>% plot(main='Distribution of noun frequency', xlab='Rank', ylab='Frequency')

In [ ]:
freq_noun %>% sort(decreasing=T) %>% plot(main='Distribution of noun frequency', xlab='Rank (log)', ylab='Frequency (log)', log='xy')

In [ ]:
# cf. Noraml Distribution
dist = rnorm(10000)
dist %>% hist(100, main='Normal distribution (n = 10000)')
dist %>% sort(decreasing=T) %>% plot(main='Normal distribution (n = 10000)', xlab='Rank', ylab='Value')

In [ ]:
# cf. Poisson Distribution
dist = rpois(10000, 1) 
dist %>% hist(main='Poisson distribution (n = 10000, lambda=1)', breaks=7)
dist %>% sort(decreasing=T) %>% plot(main='Poisson distribution (n = 10000, lambda=1)', xlab='Rank', ylab='Value')

In [ ]:
freq_noun %>% sort() %>% tail(30)

In [ ]:
# フォントの設定
par(family = "Meiryo")

In [ ]:
# Web環境のみ
library(showtext)
font_add_google("Noto Sans JP", "jpfont")  # Googleフォントから
showtext_auto()

In [ ]:
freq_noun %>% sort() %>% tail(30) %>% barplot(horiz=T, las=1, main='Top 30 nouns', xlab='Frequency')

In [ ]:
# stopwords
stopwords = c('よう', '上', '中',  'もの', 'の', 'それ', '一', '事', '何','ん', 'どこ')

In [ ]:
freq_noun %>% head()

In [ ]:
# tableなので以下の書き方にする
freq_noun[!names(freq_noun) %in% stopwords] %>% 
    sort() %>% 
    tail(30) %>% 
    barplot(horiz=T, las=1, main='Top 30 nouns', xlab='Frequency', cex.names=0.9)

### TF-IDF

$$
TF_{i,j} = \frac{n_{i,j}}{\sum{_{k}n_{k,j}}} = \frac{文書 d_j における単語 t_i  の頻度}{文書d中の総単語数}
$$

$$
IDF{i,j} = \log{\frac{|D|}{|{d:d \ni t_i}|}} = \log{(1 / \frac{ 単語t_iを含む文書数}{ 総文書数})}
$$

$$
TFIDF_{i,j} = TF_{i,j} \times IDF_{i,j}
$$


In [ ]:
# tf, idfをそれぞれ求める関数

tf = function(df){
    return(t(t(df) / colSums(df)))
}

idf = function(df){
    doc_sums = (df > 0) %>% rowSums() + 1
    return(log2(ncol(df)/doc_sums))
}

In [ ]:
count_noun %>% head()

In [ ]:
tfidf= tf(count_noun) * idf(count_noun) 


In [ ]:
tfidf %>% head()

In [ ]:
# 2行5列で表示
par(mfrow=c(2,5)) 

for(i in 1:10){
tfidf[,i] %>% 
    sort() %>% 
    tail(10)  %>%  barplot(horiz=T, las=2, main=character(i))
}

# 図表設定を初期化
par(mfrow=c(1,1)) 